<a href="https://colab.research.google.com/github/sanskkriti/2D-Array/blob/main/HateSpeechDectectionModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Data Handling
import pandas as pd
import numpy as np

# Text Preprocessing
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Model Building
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving hatespeech.csv to hatespeech.csv


In [ ]:
df = pd.read_csv('hatespeech.csv')
df.head()

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,tweet
0,0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [ ]:
# Assuming your CSV has a column named 'class' where:
# 0 = hate speech, 1 = offensive language, 2 = neither

df['label'] = df['class'].apply(lambda x: 1 if x == 0 else 0)

# Get tweet text and labels
texts = df['tweet'].tolist()
labels = df['label'].tolist()

In [ ]:
# Create vocabulary
vocab = set(word for text in texts for word in text.lower().split())
word2idx = {word: idx + 1 for idx, word in enumerate(vocab)}  # index 0 for padding

# Convert text to list of word indices
def text_to_sequence(text):
    return [word2idx.get(word, 0) for word in text.lower().split()]

sequences = [text_to_sequence(text) for text in texts]

In [ ]:
# Pad sequences to same length
maxlen = 50
padded_sequences = pad_sequences(sequences, maxlen=maxlen, padding='post')

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.2, random_state=42)

In [ ]:
model = Sequential([
    Embedding(input_dim=len(vocab)+1, output_dim=16, input_length=maxlen),
    LSTM(32),
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

model.fit(tf.convert_to_tensor(X_train), tf.convert_to_tensor(y_train),
          epochs=5, batch_size=64)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
310/310 ━━━━━━━━━━━━━━━━━━━━ 16s 37ms/step - accuracy: 0.9407 - loss: 0.2790
Epoch 2/5
310/310 ━━━━━━━━━━━━━━━━━━━━ 21s 38ms/step - accuracy: 0.9452 - loss: 0.2127
Epoch 3/5
310/310 ━━━━━━━━━━━━━━━━━━━━ 11s 36ms/step - accuracy: 0.9421 - loss: 0.2221
Epoch 4/5
310/310 ━━━━━━━━━━━━━━━━━━━━ 12s 38ms/step - accuracy: 0.9444 - loss: 0.2155
Epoch 5/5
310/310 ━━━━━━━━━━━━━━━━━━━━ 20s 35ms/step - accuracy: 0.9425 - loss: 0.2208


In [ ]:
loss, accuracy = model.evaluate(tf.convert_to_tensor(X_test), tf.convert_to_tensor(y_test))
print(f"\nTest Accuracy: {accuracy:.2f}")

155/155 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9386 - loss: 0.2309

Test Accuracy: 0.94
